# Evaluation Layer: LLM-as-Judge for the Event-Planning Multi-Agent System

This notebook adds an **evaluation layer** on top of the four single-purpose agents from
`Task.ipynb` (`VenueAgent`, `CateringLogisticsAgent`, `AVVendorCoordinationAgent`,
`AgendaAssemblyAgent`). Before any agent's output is passed downstream to the next agent (or to
the end user), it is scored by an `Evaluator` acting as an **LLM-as-judge**.

**Deliverables in this notebook**

1. An `Evaluator` class with a defined rubric (4 criteria relevant to event planning) and a
   scoring prompt that produces a structured, schema-validated verdict for every agent output.
2. Retry / escalation logic: outputs scoring below a configurable threshold trigger a retry of
   the same agent; if the retry also fails, the output is flagged for **human review** instead of
   silently continuing.
3. An evaluation report covering **five synthetic test cases** spanning all four agents, with
   per-criterion scores, an overall score, and a pass/fail verdict for each.

As in the earlier notebooks, every agent and the judge itself depend only on the duck-typed
`LLMClient` protocol, so the whole notebook runs deterministically with no network access or API
credentials via scripted fake backends. Swap in `AzureOpenAIClient` for production use.


In [1]:
import json
import os
import sys
from collections import deque
from enum import Enum
from typing import Any, Dict, List, Literal, Optional, Protocol, Type

import pandas as pd
from pydantic import BaseModel, Field

# Make the repo-root `init_creds.py` importable - only needed if/when `AzureOpenAIClient` below
# is actually instantiated.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)


class EventType(str, Enum):
    """Shared event taxonomy used as input by every agent (same as `Task.ipynb`)."""

    CONFERENCE = "conference"
    WEDDING = "wedding"
    CORPORATE_OFFSITE = "corporate_offsite"
    PRODUCT_LAUNCH = "product_launch"
    GALA = "gala"
    WORKSHOP = "workshop"
    OTHER = "other"


class LLMClient(Protocol):
    """Minimal interface every agent - and the judge itself - depends on (same contract as
    `Task.ipynb`)."""

    def generate(
        self,
        system_prompt: str,
        user_payload: Dict[str, Any],
        schema_name: str,
        json_schema: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Return a dict that should validate against `json_schema`."""
        ...


class AzureOpenAIClient:
    """Production LLM backend: Azure OpenAI chat completions with structured JSON output.

    Credentials are resolved lazily (via the repo's `init_creds.py` helpers) only when an
    instance is actually constructed, so importing/using this module never requires network
    access or secrets - that is reserved for the scripted fakes below.
    """

    def __init__(self, model: str = "gpt-4o-mini", api_version: str = "2025-04-01-preview"):
        from openai import AzureOpenAI
        import init_creds as creds

        self._client = AzureOpenAI(
            api_key=creds.get_api_key(),
            azure_endpoint=creds.get_endpoint(),
            api_version=api_version,
        )
        self._model = model

    def generate(self, system_prompt, user_payload, schema_name, json_schema):
        response = self._client.chat.completions.create(
            model=self._model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(user_payload)},
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {"name": schema_name, "schema": json_schema, "strict": False},
            },
        )
        return json.loads(response.choices[0].message.content)


class ScriptedLLMClient:
    """Deterministic fake backend: returns one canned response per call, popped off a scripted
    queue in order. Used for BOTH agents (to script a flawed first attempt followed by a fixed
    retry attempt) and the judge (to script the corresponding scores for each attempt) - the same
    philosophy as `Task2.ipynb`'s `ScriptedLLMClient`, with no network access required."""

    def __init__(self, responses: List[Dict[str, Any]]):
        self._responses = deque(responses)
        self.calls: List[Dict[str, Any]] = []

    def generate(self, system_prompt, user_payload, schema_name, json_schema):
        self.calls.append({"schema_name": schema_name, "user_payload": user_payload})
        if not self._responses:
            raise RuntimeError("ScriptedLLMClient ran out of canned responses")
        return self._responses.popleft()


## 1. The four specialist agents (reused from `Task.ipynb`)

Same `BaseAgent` scaffolding and the same four agent classes/schemas as `Task.ipynb` /
`Task3.ipynb` - repeated here so this notebook is self-contained and runnable on its own. The
evaluation layer below scores whatever structured output these agents produce.


In [2]:
class BaseAgent:
    """Shared scaffolding for single-purpose, structured-output agents."""

    name: str
    system_prompt: str
    input_model: Type[BaseModel]
    output_model: Type[BaseModel]

    def __init__(self, llm: LLMClient):
        self._llm = llm

    def run(self, request):
        if not isinstance(request, self.input_model):
            request = self.input_model.model_validate(request)
        payload = request.model_dump(mode="json")
        raw_output = self._llm.generate(
            system_prompt=self.system_prompt,
            user_payload=payload,
            schema_name=self.name,
            json_schema=self.output_model.model_json_schema(),
        )
        return self.output_model.model_validate(raw_output)


class VenueRequest(BaseModel):
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(gt=0)
    location: str = Field(min_length=1)
    event_date: Optional[str] = None
    special_requirements: List[str] = Field(default_factory=list)


class VenueOption(BaseModel):
    name: str
    area: str
    capacity: int = Field(gt=0)
    estimated_cost_usd: float = Field(ge=0)
    amenities: List[str] = Field(default_factory=list)
    rationale: str


class VenueRecommendation(BaseModel):
    recommended_venues: List[VenueOption] = Field(min_length=1)
    assumptions: List[str] = Field(default_factory=list)
    notes: Optional[str] = None


class VenueAgent(BaseAgent):
    name = "venue_recommendation_agent"
    input_model = VenueRequest
    output_model = VenueRecommendation
    system_prompt = (
        "You are a single-purpose venue recommendation specialist for live events. Propose "
        "2-4 realistic candidate venues from headcount, budget, location, and event type. Never "
        "recommend a venue with capacity below the requested headcount. Respond ONLY with the "
        "structured JSON object defined by the provided schema."
    )


class CateringRequest(BaseModel):
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(gt=0)
    meal_period: Literal["breakfast", "brunch", "lunch", "dinner", "reception"]
    dietary_restrictions: List[str] = Field(default_factory=list)
    service_style_preference: Optional[
        Literal["plated", "buffet", "family_style", "food_stations", "cocktail"]
    ] = None


class MenuItem(BaseModel):
    course: str
    description: str
    dietary_tags: List[str] = Field(default_factory=list)


class CateringPlan(BaseModel):
    service_style: str
    menu_items: List[MenuItem] = Field(min_length=1)
    staffing_notes: str
    estimated_cost_usd: float = Field(ge=0)
    cost_per_head_usd: float = Field(ge=0)
    assumptions: List[str] = Field(default_factory=list)


class CateringLogisticsAgent(BaseAgent):
    name = "catering_logistics_agent"
    input_model = CateringRequest
    output_model = CateringPlan
    system_prompt = (
        "You are a single-purpose catering and food-service logistics specialist. Propose ONE "
        "coherent catering plan covering every dietary restriction supplied, with "
        "estimated_cost_usd approximately equal to cost_per_head_usd times headcount. Respond "
        "ONLY with the structured JSON object defined by the provided schema."
    )


class AVRequest(BaseModel):
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(gt=0)
    venue_type: Optional[str] = None
    special_requirements: List[str] = Field(default_factory=list)


class AVSetupItem(BaseModel):
    category: str
    description: str
    quantity: int = Field(ge=1)


class VendorSuggestion(BaseModel):
    vendor_role: str
    notes: str


class AVPlan(BaseModel):
    av_setup: List[AVSetupItem] = Field(min_length=1)
    recommended_vendors: List[VendorSuggestion] = Field(min_length=1)
    estimated_cost_usd: float = Field(ge=0)
    setup_lead_time_hours: float = Field(ge=0)
    assumptions: List[str] = Field(default_factory=list)


class AVVendorCoordinationAgent(BaseAgent):
    name = "av_vendor_coordination_agent"
    input_model = AVRequest
    output_model = AVPlan
    system_prompt = (
        "You are a single-purpose audiovisual and vendor-coordination specialist. Propose ONE "
        "AV setup plan and the vendors needed to deliver it, covering every special requirement "
        "and respecting the stated budget. Respond ONLY with the structured JSON object defined "
        "by the provided schema."
    )


class AgendaRequest(BaseModel):
    event_name: str
    event_date: str
    start_time: str
    end_time: str
    venue: VenueRecommendation
    catering: CateringPlan
    av: AVPlan


class RunOfShowItem(BaseModel):
    start_time: str
    end_time: str
    activity: str
    owner: str
    location: Optional[str] = None
    notes: Optional[str] = None


class RunOfShow(BaseModel):
    event_name: str
    event_date: str
    selected_venue: str
    run_of_show: List[RunOfShowItem] = Field(min_length=1)
    overall_notes: List[str] = Field(default_factory=list)


class AgendaAssemblyAgent(BaseAgent):
    name = "agenda_assembly_agent"
    input_model = AgendaRequest
    output_model = RunOfShow
    system_prompt = (
        "You are a single-purpose agenda-assembly specialist. Choose exactly ONE venue from "
        "`venue.recommended_venues` (naming it exactly in `selected_venue`) and sequence "
        "catering + AV + main activities into a single, non-overlapping chronological "
        "run-of-show. Respond ONLY with the structured JSON object defined by the provided "
        "schema."
    )


## 2. Evaluation rubric and the `Evaluator` (LLM-as-judge)

The judge scores every agent output against **four** criteria relevant to event planning:

| Criterion | What it checks |
|---|---|
| `coherence` | Internal consistency - do numbers, names, and references agree (e.g. `cost_per_head_usd * headcount ~= estimated_cost_usd`, the same venue name used throughout, no overlapping schedule entries)? |
| `completeness` | Does the output address every requirement in the request (every dietary restriction covered by a menu item, every special requirement covered by an AV/vendor entry, no generic placeholder content)? |
| `budget_adherence` | Does the proposed cost respect `budget_usd`? If it can't, is that explicitly disclosed in `assumptions` rather than silently exceeded? |
| `feasibility` | Is the plan realistic and actionable given hard constraints (e.g. venue `capacity >= headcount`, plausible staffing/lead times)? |

Design choices worth calling out:
- The judge LLM returns **raw per-criterion scores + an overall score + a rationale** (a
  `JudgeVerdict`) - it does **not** decide pass/fail itself. `Evaluator` applies the
  `pass_threshold` deterministically in code, so the retry/escalation decision never depends on
  the judge reliably following a "decide pass/fail" instruction.
- `overall_score` is holistic (not a plain average) so the judge can weight the criteria that
  matter most for a given output - the prompt tells it to weight `budget_adherence` and
  `completeness` most heavily, since silent budget overruns and missing requirements are the
  costliest failure modes in this pipeline.


In [3]:
RUBRIC: Dict[str, str] = {
    "coherence": (
        "Internal consistency - do numbers, names, and references agree with each other "
        "(e.g. cost_per_head_usd * headcount ~= estimated_cost_usd, the same venue name is used "
        "throughout, and there are no overlapping or contradictory schedule entries)?"
    ),
    "completeness": (
        "Does the output address every requirement in the request - every dietary restriction "
        "covered by a menu item, every special_requirement covered by an av_setup entry or "
        "vendor, and every required field populated with specific, non-generic content?"
    ),
    "budget_adherence": (
        "Does the proposed cost respect the requester's budget_usd? If the plan cannot stay "
        "within budget, is that explicitly and clearly disclosed in `assumptions` rather than "
        "silently exceeded?"
    ),
    "feasibility": (
        "Is the plan realistic and actionable given hard constraints (e.g. venue capacity >= "
        "headcount, staffing/lead times are plausible for the scale of the event, vendor/AV "
        "choices make sense for the event type)?"
    ),
}


class CriterionScore(BaseModel):
    criterion: str
    score: float = Field(ge=0, le=10, description="0 = completely fails this criterion, 10 = fully satisfies it")
    explanation: str = Field(description="Concrete evidence from the input/output that justifies the score")


class JudgeVerdict(BaseModel):
    """Raw structured output returned by the judge LLM - scores only, no pass/fail decision."""

    criterion_scores: List[CriterionScore] = Field(min_length=4)
    overall_score: float = Field(ge=0, le=10, description="Holistic 0-10 readiness score")
    summary: str = Field(description="1-3 sentence justification of the overall verdict")


class EvaluationResult(BaseModel):
    """Final evaluation record: the judge's structured verdict plus the evaluator's own
    deterministic pass/fail + routing decision (never delegated to the LLM)."""

    agent_name: str
    attempt: int
    criterion_scores: List[CriterionScore]
    overall_score: float
    passed: bool
    recommendation: Literal["accept", "retry", "escalate_to_human"]
    summary: str


class Evaluator:
    """LLM-as-judge: scores a single agent's output against a fixed rubric before it is passed
    downstream to the next agent (or to the end user)."""

    def __init__(self, llm: LLMClient, rubric: Dict[str, str] = RUBRIC, pass_threshold: float = 7.0):
        self._llm = llm
        self.rubric = rubric
        self.pass_threshold = pass_threshold

    def _system_prompt(self) -> str:
        criteria_lines = "\n".join(f"- {name} (0-10): {desc}" for name, desc in self.rubric.items())
        return (
            "You are an impartial quality-assurance judge for an event-planning multi-agent "
            "system. You are given the JSON request sent to one specialist agent and the JSON "
            "output it produced. Score that output against EACH of the following criteria on a "
            "0-10 scale, citing concrete evidence from the input/output in your explanation:\n"
            f"{criteria_lines}\n\n"
            "Then give one holistic `overall_score` (0-10) reflecting the output's overall "
            "readiness to be passed downstream to the next agent - weight `budget_adherence` and "
            "`completeness` most heavily, since silent budget overruns or missing requirements "
            "are the most costly failure modes in this pipeline. Do not soften scores out of "
            "politeness - a plan that violates a hard constraint (e.g. venue capacity below "
            "headcount) must score low on the relevant criterion.\n"
            "Respond ONLY with the structured JSON object defined by the provided schema."
        )

    def evaluate(
        self,
        agent_name: str,
        agent_input: Dict[str, Any],
        agent_output: Dict[str, Any],
        attempt: int = 1,
    ) -> EvaluationResult:
        raw = self._llm.generate(
            system_prompt=self._system_prompt(),
            user_payload={
                "agent_name": agent_name,
                "agent_input": agent_input,
                "agent_output": agent_output,
            },
            schema_name="evaluation_judge",
            json_schema=JudgeVerdict.model_json_schema(),
        )
        verdict = JudgeVerdict.model_validate(raw)
        passed = verdict.overall_score >= self.pass_threshold
        return EvaluationResult(
            agent_name=agent_name,
            attempt=attempt,
            criterion_scores=verdict.criterion_scores,
            overall_score=verdict.overall_score,
            passed=passed,
            recommendation="accept" if passed else "retry",
            summary=verdict.summary,
        )


## 3. Retry / escalation logic

`run_agent_with_evaluation` wraps a single agent call with the evaluator:

1. Run the agent, then score the output.
2. If `overall_score >= pass_threshold`, accept it immediately.
3. Otherwise, **retry** the same agent (up to `max_retries` times).
4. If every attempt - including all retries - still falls below the threshold, the output is
   **flagged for human review** (`recommendation = "escalate_to_human"`) instead of being passed
   downstream silently.

The full per-attempt evaluation history is kept on the returned `AgentRunResult`, so the final
report can show *why* a retry was triggered and whether it actually fixed the problem.


In [4]:
class AgentRunResult(BaseModel):
    """Outcome of running one agent through the evaluation/retry/escalation pipeline."""

    agent_name: str
    attempts: int
    final_output: Dict[str, Any]
    final_evaluation: EvaluationResult
    evaluation_history: List[EvaluationResult]
    escalated: bool


def run_agent_with_evaluation(
    agent: BaseAgent,
    request: BaseModel,
    evaluator: Evaluator,
    max_retries: int = 1,
) -> AgentRunResult:
    """Run `agent` on `request`, score every attempt with `evaluator`, and retry whenever the
    score falls below the evaluator's pass threshold. If the output is still failing after
    `max_retries` retries, escalate it for human review rather than silently forwarding a
    low-quality result downstream."""
    history: List[EvaluationResult] = []
    output = None

    for attempt in range(1, max_retries + 2):  # initial attempt + up to max_retries retries
        output = agent.run(request)
        evaluation = evaluator.evaluate(
            agent_name=agent.name,
            agent_input=request.model_dump(mode="json"),
            agent_output=output.model_dump(mode="json"),
            attempt=attempt,
        )
        history.append(evaluation)
        if evaluation.passed:
            return AgentRunResult(
                agent_name=agent.name,
                attempts=attempt,
                final_output=output.model_dump(mode="json"),
                final_evaluation=evaluation,
                evaluation_history=history,
                escalated=False,
            )

    escalated_evaluation = history[-1].model_copy(update={"recommendation": "escalate_to_human"})
    history[-1] = escalated_evaluation
    return AgentRunResult(
        agent_name=agent.name,
        attempts=max_retries + 1,
        final_output=output.model_dump(mode="json"),
        final_evaluation=escalated_evaluation,
        evaluation_history=history,
        escalated=True,
    )


## 4. Five synthetic test cases

Each test case scripts the agent's attempt(s) via `ScriptedLLMClient` (a flawed first attempt,
optionally followed by a fixed retry attempt) and scripts the judge's corresponding verdict(s) for
each attempt, so the whole report below is fully deterministic - no network access required.

| # | Agent | Scenario | Expected outcome |
|---|---|---|---|
| 1 | `VenueAgent` | Venue fits headcount/budget, wheelchair accessibility covered | **Pass** on attempt 1 |
| 2 | `CateringLogisticsAgent` | Attempt 1 drops the dietary tags and its cost math doesn't add up | **Retry -> Pass** |
| 3 | `AVVendorCoordinationAgent` | Attempt 1 ignores the livestream requirement and silently doubles the budget; attempt 2 still doesn't fix either | **Retry -> Escalate to human review** |
| 4 | `AgendaAssemblyAgent` | Attempt 1 has overlapping time slots and invents a venue not in the venue agent's list | **Retry -> Pass** |
| 5 | `VenueAgent` | Attempt 1 recommends a venue whose capacity is below the requested headcount (a hard constraint violation) | **Retry -> Pass** |


In [5]:
def make_verdict(scores: Dict[str, float], explanations: Dict[str, str], overall: float, summary: str) -> Dict[str, Any]:
    """Build a `JudgeVerdict`-shaped dict from per-criterion scores/explanations - a small helper
    to keep the five test-case definitions below readable."""
    return {
        "criterion_scores": [
            {"criterion": name, "score": score, "explanation": explanations[name]}
            for name, score in scores.items()
        ],
        "overall_score": overall,
        "summary": summary,
    }


# --- Test case 1: VenueAgent - good output, passes on attempt 1 --------------------------------
tc1_request = VenueRequest(
    event_type=EventType.CONFERENCE,
    headcount=150,
    budget_usd=12000,
    location="Austin, TX",
    event_date="2026-03-10",
    special_requirements=["wheelchair accessible"],
)
tc1_agent_llm = ScriptedLLMClient(responses=[
    {
        "recommended_venues": [
            {
                "name": "Lakeside Conference Center",
                "area": "Downtown Austin",
                "capacity": 180,
                "estimated_cost_usd": 9500.0,
                "amenities": ["wifi", "wheelchair ramp", "on-site parking"],
                "rationale": "Capacity comfortably exceeds headcount and cost is well under budget.",
            }
        ],
        "assumptions": ["Pricing assumes a weekday booking."],
        "notes": None,
    }
])
tc1_judge_llm = ScriptedLLMClient(responses=[
    make_verdict(
        scores={"coherence": 9.0, "completeness": 9.5, "budget_adherence": 9.5, "feasibility": 9.0},
        explanations={
            "coherence": "Single venue, capacity/cost/amenities are all mutually consistent.",
            "completeness": "Wheelchair accessibility explicitly covered via the 'wheelchair ramp' amenity.",
            "budget_adherence": "$9,500 estimated cost is comfortably under the $12,000 budget.",
            "feasibility": "Capacity (180) exceeds headcount (150); no hard constraints violated.",
        },
        overall=9.2,
        summary="Well within budget, capacity, and accessibility requirements - ready to pass downstream.",
    )
])


In [6]:
# --- Test case 2: CateringLogisticsAgent - flawed attempt 1, fixed attempt 2 -------------------
tc2_request = CateringRequest(
    event_type=EventType.CONFERENCE,
    headcount=150,
    budget_usd=8000,
    meal_period="lunch",
    dietary_restrictions=["vegetarian", "gluten-free"],
)
tc2_agent_llm = ScriptedLLMClient(responses=[
    {  # attempt 1: no dietary coverage at all, and cost_per_head * headcount != estimated_cost
        "service_style": "buffet",
        "menu_items": [
            {"course": "main", "description": "Herb-roasted chicken", "dietary_tags": []},
        ],
        "staffing_notes": "2 chefs, 4 servers for 150 guests.",
        "estimated_cost_usd": 7500.0,
        "cost_per_head_usd": 70.0,  # 70 * 150 = 10,500 != 7,500
        "assumptions": [],
    },
    {  # attempt 2: vegetarian/gluten-free covered, cost math fixed
        "service_style": "buffet",
        "menu_items": [
            {"course": "main", "description": "Herb-roasted chicken", "dietary_tags": ["gluten-free"]},
            {"course": "main", "description": "Grilled vegetable & quinoa stack", "dietary_tags": ["vegetarian", "gluten-free"]},
        ],
        "staffing_notes": "2 chefs, 4 servers for 150 guests.",
        "estimated_cost_usd": 7500.0,
        "cost_per_head_usd": 50.0,  # 50 * 150 = 7,500 - consistent
        "assumptions": [],
    },
])
tc2_judge_llm = ScriptedLLMClient(responses=[
    make_verdict(
        scores={"coherence": 4.0, "completeness": 3.0, "budget_adherence": 6.0, "feasibility": 6.0},
        explanations={
            "coherence": "cost_per_head_usd (70) * headcount (150) = 10,500, which contradicts estimated_cost_usd (7,500).",
            "completeness": "Neither 'vegetarian' nor 'gluten-free' is covered by any menu_item's dietary_tags.",
            "budget_adherence": "Stated total (7,500) is within the 8,000 budget, but the figure is internally inconsistent.",
            "feasibility": "Staffing plan is plausible for 150 guests, but the menu itself is incomplete.",
        },
        overall=4.3,
        summary="Fails to cover required dietary restrictions and the cost figures don't reconcile - not safe to pass downstream.",
    ),
    make_verdict(
        scores={"coherence": 9.0, "completeness": 9.0, "budget_adherence": 9.0, "feasibility": 8.5},
        explanations={
            "coherence": "cost_per_head_usd (50) * headcount (150) = 7,500, matching estimated_cost_usd exactly.",
            "completeness": "Both vegetarian and gluten-free are now covered by at least one menu item's dietary_tags.",
            "budget_adherence": "7,500 total is within the 8,000 budget with no silent overrun.",
            "feasibility": "Staffing plan remains realistic for a 150-guest buffet lunch.",
        },
        overall=8.9,
        summary="Dietary coverage and cost math are now consistent and within budget - ready to pass downstream.",
    ),
])


In [7]:
# --- Test case 3: AVVendorCoordinationAgent - both attempts fail -> escalate -------------------
tc3_request = AVRequest(
    event_type=EventType.CONFERENCE,
    headcount=150,
    budget_usd=5000,
    venue_type="ballroom",
    special_requirements=["livestream"],
)
tc3_agent_llm = ScriptedLLMClient(responses=[
    {  # attempt 1: no livestream provider at all, and silently ~2x over budget
        "av_setup": [
            {"category": "audio", "description": "Wireless handheld microphones", "quantity": 2},
        ],
        "recommended_vendors": [
            {"vendor_role": "AV production company", "notes": "Local full-service AV vendor."},
        ],
        "estimated_cost_usd": 9800.0,
        "setup_lead_time_hours": 3.0,
        "assumptions": [],
    },
    {  # attempt 2: adds a video item but still no dedicated livestream vendor, still over budget
        "av_setup": [
            {"category": "audio", "description": "Wireless handheld microphones", "quantity": 2},
            {"category": "video", "description": "Confidence monitor", "quantity": 1},
        ],
        "recommended_vendors": [
            {"vendor_role": "AV production company", "notes": "Local full-service AV vendor."},
        ],
        "estimated_cost_usd": 9200.0,
        "setup_lead_time_hours": 3.0,
        "assumptions": [],
    },
])
tc3_judge_llm = ScriptedLLMClient(responses=[
    make_verdict(
        scores={"coherence": 5.0, "completeness": 2.0, "budget_adherence": 1.0, "feasibility": 5.0},
        explanations={
            "coherence": "Setup and vendor list are at least internally consistent with each other.",
            "completeness": "'livestream' special requirement has no corresponding av_setup entry or vendor.",
            "budget_adherence": "estimated_cost_usd (9,800) is nearly double the 5,000 budget with no disclosure in assumptions.",
            "feasibility": "Lead time is plausible, but the plan does not actually deliver what was requested.",
        },
        overall=3.1,
        summary="Missing the livestream requirement entirely and silently blows past the budget - not safe to pass downstream.",
    ),
    make_verdict(
        scores={"coherence": 5.5, "completeness": 4.0, "budget_adherence": 2.0, "feasibility": 5.0},
        explanations={
            "coherence": "Still internally consistent, no new contradictions introduced.",
            "completeness": "Adding a 'confidence monitor' does not satisfy 'livestream' - no encoder/streaming provider is present.",
            "budget_adherence": "estimated_cost_usd (9,200) is still ~84% over the 5,000 budget with no disclosure.",
            "feasibility": "Plan remains executable but still does not deliver the requested capability within budget.",
        },
        overall=3.9,
        summary="Retry did not fix either root cause (missing livestream vendor, undisclosed budget overrun) - requires human review.",
    ),
])


In [8]:
# --- Test case 4: AgendaAssemblyAgent - overlapping schedule + invented venue, fixed on retry ---
tc4_venue = VenueRecommendation(
    recommended_venues=[
        VenueOption(
            name="Riverside Pavilion",
            area="Riverside District",
            capacity=220,
            estimated_cost_usd=11000.0,
            amenities=["wifi", "built-in stage"],
            rationale="Exceeds headcount while staying within budget.",
        )
    ],
    assumptions=[],
)
tc4_catering = CateringPlan(
    service_style="food_stations",
    menu_items=[MenuItem(course="main", description="Chef-attended pasta station", dietary_tags=["vegetarian"])],
    staffing_notes="3 chefs, 6 servers for 200 guests.",
    estimated_cost_usd=9000.0,
    cost_per_head_usd=45.0,
)
tc4_av = AVPlan(
    av_setup=[AVSetupItem(category="audio", description="Line array speakers", quantity=2)],
    recommended_vendors=[VendorSuggestion(vendor_role="AV production company", notes="Handles rigging and sound.")],
    estimated_cost_usd=5200.0,
    setup_lead_time_hours=4.0,
)
tc4_request = AgendaRequest(
    event_name="Product Launch Night",
    event_date="2026-05-01",
    start_time="18:00",
    end_time="22:00",
    venue=tc4_venue,
    catering=tc4_catering,
    av=tc4_av,
)
tc4_agent_llm = ScriptedLLMClient(responses=[
    {  # attempt 1: overlapping time slots, and selected_venue is not the venue it was given
        "event_name": "Product Launch Night",
        "event_date": "2026-05-01",
        "selected_venue": "Grand Ballroom",  # not in venue.recommended_venues at all
        "run_of_show": [
            {"start_time": "18:00", "end_time": "20:00", "activity": "Guest arrival and reception", "owner": "Host", "location": None, "notes": None},
            {"start_time": "19:00", "end_time": "21:00", "activity": "Food station service", "owner": "Catering team", "location": None, "notes": None},
        ],
        "overall_notes": [],
    },
    {  # attempt 2: fixed - correct venue name, non-overlapping chronological schedule
        "event_name": "Product Launch Night",
        "event_date": "2026-05-01",
        "selected_venue": "Riverside Pavilion",
        "run_of_show": [
            {"start_time": "14:00", "end_time": "18:00", "activity": "AV setup", "owner": "AV crew", "location": "Riverside Pavilion", "notes": None},
            {"start_time": "18:00", "end_time": "19:00", "activity": "Guest arrival and reception", "owner": "Host", "location": "Riverside Pavilion", "notes": None},
            {"start_time": "19:00", "end_time": "20:00", "activity": "Food station service", "owner": "Catering team", "location": "Riverside Pavilion", "notes": None},
        ],
        "overall_notes": ["Run-of-show assembled from venue, catering, and AV agent outputs."],
    },
])
tc4_judge_llm = ScriptedLLMClient(responses=[
    make_verdict(
        scores={"coherence": 3.0, "completeness": 6.0, "budget_adherence": 7.0, "feasibility": 4.0},
        explanations={
            "coherence": "'Grand Ballroom' does not match the only venue supplied ('Riverside Pavilion'), and two activities overlap from 19:00-20:00.",
            "completeness": "Catering and a reception activity are both present, though AV setup is missing from the schedule.",
            "budget_adherence": "No budget field on this input; nothing contradicts the upstream cost figures.",
            "feasibility": "Overlapping activities in the same timeframe are not actually executable as scheduled.",
        },
        overall=4.8,
        summary="Invents a venue not provided upstream and double-books 19:00-20:00 - not safe to pass downstream.",
    ),
    make_verdict(
        scores={"coherence": 9.0, "completeness": 9.0, "budget_adherence": 8.0, "feasibility": 9.0},
        explanations={
            "coherence": "selected_venue exactly matches the only venue in venue.recommended_venues, and the schedule is strictly sequential with no overlaps.",
            "completeness": "AV setup, guest reception, and catering service are all represented in the run_of_show.",
            "budget_adherence": "No budget field on this input; cost figures from upstream agents are carried through unchanged.",
            "feasibility": "Each activity has a distinct, non-overlapping time window and a clear owner.",
        },
        overall=8.9,
        summary="Schedule is now internally consistent, non-overlapping, and uses the correct venue - ready to pass downstream.",
    ),
])


In [9]:
# --- Test case 5: VenueAgent - capacity below headcount (hard constraint violation), fixed on retry
tc5_request = VenueRequest(
    event_type=EventType.GALA,
    headcount=300,
    budget_usd=50000,
    location="Miami, FL",
    special_requirements=["waterfront"],
)
tc5_agent_llm = ScriptedLLMClient(responses=[
    {  # attempt 1: capacity (250) is below the requested headcount (300)
        "recommended_venues": [
            {
                "name": "Bayview Hall",
                "area": "Downtown Miami",
                "capacity": 250,
                "estimated_cost_usd": 42000.0,
                "amenities": ["waterfront view"],
                "rationale": "Waterfront location within budget.",
            }
        ],
        "assumptions": [],
        "notes": None,
    },
    {  # attempt 2: capacity now exceeds headcount
        "recommended_venues": [
            {
                "name": "Bayview Grand Hall",
                "area": "Downtown Miami",
                "capacity": 340,
                "estimated_cost_usd": 46000.0,
                "amenities": ["waterfront view", "valet parking"],
                "rationale": "Larger waterfront hall comfortably exceeds the 300-guest headcount within budget.",
            }
        ],
        "assumptions": [],
        "notes": None,
    },
])
tc5_judge_llm = ScriptedLLMClient(responses=[
    make_verdict(
        scores={"coherence": 6.0, "completeness": 6.0, "budget_adherence": 8.0, "feasibility": 2.0},
        explanations={
            "coherence": "Single venue, internally consistent fields.",
            "completeness": "Waterfront requirement is covered, but no other gap analysis is possible with one option.",
            "budget_adherence": "42,000 is within the 50,000 budget.",
            "feasibility": "Capacity (250) is BELOW the requested headcount (300) - a hard constraint violation.",
        },
        overall=4.6,
        summary="Violates the hard capacity >= headcount constraint - cannot be passed downstream as-is.",
    ),
    make_verdict(
        scores={"coherence": 9.0, "completeness": 8.5, "budget_adherence": 8.5, "feasibility": 9.0},
        explanations={
            "coherence": "Single venue, internally consistent fields.",
            "completeness": "Waterfront requirement remains covered.",
            "budget_adherence": "46,000 is within the 50,000 budget.",
            "feasibility": "Capacity (340) now comfortably exceeds the 300-guest headcount.",
        },
        overall=8.8,
        summary="Capacity constraint is now satisfied and the venue remains within budget - ready to pass downstream.",
    ),
])


## 5. Run the evaluator across all five test cases


In [10]:
TEST_CASES = [
    {
        "id": "TC1",
        "description": "VenueAgent - good output (capacity, budget, accessibility all satisfied)",
        "agent": VenueAgent(tc1_agent_llm),
        "request": tc1_request,
        "evaluator": Evaluator(tc1_judge_llm),
    },
    {
        "id": "TC2",
        "description": "CateringLogisticsAgent - missing dietary coverage + cost mismatch",
        "agent": CateringLogisticsAgent(tc2_agent_llm),
        "request": tc2_request,
        "evaluator": Evaluator(tc2_judge_llm),
    },
    {
        "id": "TC3",
        "description": "AVVendorCoordinationAgent - missing livestream vendor + silent budget overrun",
        "agent": AVVendorCoordinationAgent(tc3_agent_llm),
        "request": tc3_request,
        "evaluator": Evaluator(tc3_judge_llm),
    },
    {
        "id": "TC4",
        "description": "AgendaAssemblyAgent - overlapping schedule + invented venue",
        "agent": AgendaAssemblyAgent(tc4_agent_llm),
        "request": tc4_request,
        "evaluator": Evaluator(tc4_judge_llm),
    },
    {
        "id": "TC5",
        "description": "VenueAgent - venue capacity below requested headcount",
        "agent": VenueAgent(tc5_agent_llm),
        "request": tc5_request,
        "evaluator": Evaluator(tc5_judge_llm),
    },
]

run_results: Dict[str, AgentRunResult] = {}
for case in TEST_CASES:
    result = run_agent_with_evaluation(case["agent"], case["request"], case["evaluator"], max_retries=1)
    run_results[case["id"]] = result
    print(
        f"{case['id']} ({case['agent'].name}): attempts={result.attempts}, "
        f"final_score={result.final_evaluation.overall_score:.1f}, "
        f"recommendation={result.final_evaluation.recommendation}"
    )


TC1 (venue_recommendation_agent): attempts=1, final_score=9.2, recommendation=accept
TC2 (catering_logistics_agent): attempts=2, final_score=8.9, recommendation=accept
TC3 (av_vendor_coordination_agent): attempts=2, final_score=3.9, recommendation=escalate_to_human
TC4 (agenda_assembly_agent): attempts=2, final_score=8.9, recommendation=accept
TC5 (venue_recommendation_agent): attempts=2, final_score=8.8, recommendation=accept


## 6. Evaluation report

For each test case: every attempt's overall score (showing the retry's effect where triggered),
and the final pass/fail verdict with the judge's per-criterion scores and rationale for the
**last** attempt.


In [11]:
def verdict_label(result: AgentRunResult) -> str:
    if result.escalated:
        return "FAIL -> ESCALATED"
    if result.attempts > 1:
        return "PASS (on retry)"
    return "PASS"


report_rows = []
for case in TEST_CASES:
    result = run_results[case["id"]]
    scores_by_criterion = {cs.criterion: cs.score for cs in result.final_evaluation.criterion_scores}
    attempt_scores = " -> ".join(f"{e.overall_score:.1f}" for e in result.evaluation_history)
    report_rows.append({
        "Test Case": case["id"],
        "Agent": case["agent"].name,
        "Scenario": case["description"],
        "Attempts": result.attempts,
        "Score per attempt": attempt_scores,
        "Coherence": scores_by_criterion.get("coherence"),
        "Completeness": scores_by_criterion.get("completeness"),
        "Budget Adherence": scores_by_criterion.get("budget_adherence"),
        "Feasibility": scores_by_criterion.get("feasibility"),
        "Final Score": round(result.final_evaluation.overall_score, 1),
        "Result": verdict_label(result),
        "Explanation": result.final_evaluation.summary,
    })

report_df = pd.DataFrame(report_rows)
pd.set_option("display.max_colwidth", 80)
report_df


,Test Case,Agent,Scenario,Attempts,Score per attempt,Coherence,Completeness,Budget Adherence,Feasibility,Final Score,Result,Explanation
0,TC1,venue_recommendation_agent,"VenueAgent - good output (capacity, budget, accessibility all satisfied)",1,9.2,9.0,9.5,9.5,9.0,9.2,PASS,"Well within budget, capacity, and accessibility requirements - ready to pass..."
1,TC2,catering_logistics_agent,CateringLogisticsAgent - missing dietary coverage + cost mismatch,2,4.3 -> 8.9,9.0,9.0,9.0,8.5,8.9,PASS (on retry),Dietary coverage and cost math are now consistent and within budget - ready ...
2,TC3,av_vendor_coordination_agent,AVVendorCoordinationAgent - missing livestream vendor + silent budget overrun,2,3.1 -> 3.9,5.5,4.0,2.0,5.0,3.9,FAIL -> ESCALATED,"Retry did not fix either root cause (missing livestream vendor, undisclosed ..."
3,TC4,agenda_assembly_agent,AgendaAssemblyAgent - overlapping schedule + invented venue,2,4.8 -> 8.9,9.0,9.0,8.0,9.0,8.9,PASS (on retry),"Schedule is now internally consistent, non-overlapping, and uses the correct..."
4,TC5,venue_recommendation_agent,VenueAgent - venue capacity below requested headcount,2,4.6 -> 8.8,9.0,8.5,8.5,9.0,8.8,PASS (on retry),Capacity constraint is now satisfied and the venue remains within budget - r...


In [12]:
pass_count = sum(1 for r in run_results.values() if not r.escalated)
escalated_count = sum(1 for r in run_results.values() if r.escalated)
retried_count = sum(1 for r in run_results.values() if r.attempts > 1)

print(f"Pass: {pass_count}/5   Escalated to human review: {escalated_count}/5   Retried: {retried_count}/5")

# Sanity checks on the retry/escalation wiring itself.
assert run_results["TC1"].attempts == 1 and not run_results["TC1"].escalated, "TC1 should pass on the first attempt"
assert run_results["TC2"].attempts == 2 and not run_results["TC2"].escalated, "TC2 should pass only after a retry"
assert run_results["TC3"].escalated and run_results["TC3"].final_evaluation.recommendation == "escalate_to_human", (
    "TC3 should exhaust its retry and be escalated for human review"
)
assert run_results["TC4"].attempts == 2 and not run_results["TC4"].escalated, "TC4 should pass only after a retry"
assert run_results["TC5"].attempts == 2 and not run_results["TC5"].escalated, "TC5 should pass only after a retry"
print("\nAll retry/escalation routing checks passed.")


Pass: 4/5   Escalated to human review: 1/5   Retried: 4/5

All retry/escalation routing checks passed.


## Summary

| Deliverable | Where |
|---|---|
| `Evaluator` class with rubric criteria + scoring prompt | `RUBRIC`, `JudgeVerdict`, `EvaluationResult`, `Evaluator` (Section 2) |
| Retry / escalation logic below threshold | `run_agent_with_evaluation` (Section 3) - retries once, then sets `recommendation = "escalate_to_human"` if still failing |
| Evaluation report for five test cases | `report_df` (Section 6), with per-criterion scores, final score, pass/fail, and explanation |

**Result distribution across the five synthetic test cases:** 4/5 ultimately pass (one on the
first attempt, three only after a retry fixed the flaw the judge caught), and 1/5 (`TC3`, the AV
agent that never adds a livestream vendor or discloses its budget overrun) is escalated for human
review after exhausting its retry - demonstrating that the evaluation layer does not let a
persistently low-quality output through silently.
